# Latent Space Geometry

High-dimensional embedding spaces develop rich geometric structure — cluster
separation, anisotropic covariance, overlapping manifolds — that is invisible
in raw form.  **GeoLatent** renders this structure as interactive 3-D scenes
using PCA, t-SNE, or UMAP projection with Mahalanobis confidence ellipsoids
and convex hulls.

Three synthetic domains are shown below, each modelling a different real-world
embedding scenario.

In [ ]:
!pip install -q "geolatent[umap]"
# After first run: Runtime > Restart session, then re-run from the next cell

In [ ]:
import numpy as np
import plotly.io as pio
from geolatent import inspect_latent_space, DARK_SCIENTIFIC

pio.renderers.default = "colab"
rng = np.random.default_rng(42)

---
## Document Embeddings — 4 Topic Clusters

Simulates 64-dimensional sentence encoder outputs (e.g., MiniLM, all-mpnet)
for four topic categories.  Signal occupies the first three dimensions;
remaining dimensions are low-variance noise — matching the structure of
real transformer embeddings after dominant directions are extracted.

Setting `scale_input=False` lets PCA find the signal dimensions directly
without equalising variance across all 64 axes.

In [ ]:
D = 64
topic_means = [
    np.array([0.0, 0.0, 0.0] + [0.0] * (D - 3)),
    np.array([8.0, 0.0, 0.0] + [0.0] * (D - 3)),
    np.array([0.0, 8.0, 0.0] + [0.0] * (D - 3)),
    np.array([4.0, 4.0, 6.0] + [0.0] * (D - 3)),
]
noise = np.full(D, 0.15)
noise[:3] = 0.7

docs = np.vstack([rng.normal(size=(200, D)) * noise + m for m in topic_means])
doc_labels = np.repeat([0, 1, 2, 3], 200)
doc_names = {0: "Science", 1: "Politics", 2: "Arts", 3: "Sport"}

cfg = DARK_SCIENTIFIC.copy()
cfg.projection.scale_input = False

for method in ("pca", "tsne", "umap"):
    inspect_latent_space(
        docs, doc_labels,
        config=cfg.with_method(method),
        show_ellipsoids=True,
        show_convex_hulls=(method == "pca"),
        class_names=doc_names,
        title=f"Document Embeddings — {method.upper()}",
    ).show()

---
## Protein Feature Vectors — Varying Cluster Compactness

Simulates 128-D biochemical feature vectors (e.g., ESM protein embeddings)
for five protein families.  Each family has a different covariance structure:
some tightly packed (low variance), others diffuse (high variance).  The
ellipsoids make this heteroscedasticity immediately visible.

In [ ]:
D = 128
families = [
    {"mean": np.array([10, 0,  0]  + [0] * (D - 3), dtype=float), "std": 0.4, "n": 120},
    {"mean": np.array([0,  10, 0]  + [0] * (D - 3), dtype=float), "std": 1.8, "n": 120},
    {"mean": np.array([0,  0,  10] + [0] * (D - 3), dtype=float), "std": 0.6, "n": 120},
    {"mean": np.array([7,  7,  0]  + [0] * (D - 3), dtype=float), "std": 2.5, "n": 120},
    {"mean": np.array([5,  0,  8]  + [0] * (D - 3), dtype=float), "std": 1.0, "n": 120},
]

bg_noise = np.full(D, 0.12)
bg_noise[:3] = 1.0

proteins = np.vstack([
    rng.normal(size=(f["n"], D)) * bg_noise * f["std"] + f["mean"]
    for f in families
])
prot_labels = np.repeat(range(5), 120)
prot_names = {0: "Kinase", 1: "Receptor", 2: "Channel", 3: "Protease", 4: "Chaperone"}

cfg2 = DARK_SCIENTIFIC.copy()
cfg2.projection.scale_input = False

inspect_latent_space(
    proteins, prot_labels,
    config=cfg2.with_method("pca"),
    show_ellipsoids=True,
    show_convex_hulls=True,
    class_names=prot_names,
    ellipsoid_confidence=0.80,
    title="Protein Family Embeddings — PCA (heterogeneous covariance)",
).show()

inspect_latent_space(
    proteins, prot_labels,
    config=cfg2.with_method("umap"),
    show_ellipsoids=True,
    class_names=prot_names,
    title="Protein Family Embeddings — UMAP",
).show()

---
## Market Regime Vectors — Non-Gaussian Structure

Simulates 32-D state vectors extracted from financial time-series windows
(e.g., returns, volatility, correlation features) for four market regimes.
Regimes are not cleanly Gaussian — two of them share overlap in the projected
space, reflecting real ambiguity at regime boundaries.

In [ ]:
D = 32
regimes = [
    {"mean": np.array([5, 0, 0]  + [0] * (D - 3), dtype=float), "std": 0.8,  "n": 180, "name": "Bull"},
    {"mean": np.array([-5, 0, 0] + [0] * (D - 3), dtype=float), "std": 1.5,  "n": 180, "name": "Bear"},
    {"mean": np.array([0, 4, 3]  + [0] * (D - 3), dtype=float), "std": 2.2,  "n": 180, "name": "Sideways"},
    {"mean": np.array([0, 0, -5] + [0] * (D - 3), dtype=float), "std": 3.5,  "n": 180, "name": "Crisis"},
]

bg = np.full(D, 0.3)
bg[:3] = 1.0

states = np.vstack([
    rng.normal(size=(r["n"], D)) * bg * r["std"] + r["mean"]
    for r in regimes
])
state_labels = np.repeat(range(4), 180)
state_names = {i: r["name"] for i, r in enumerate(regimes)}

cfg3 = DARK_SCIENTIFIC.copy()
cfg3.projection.scale_input = False

for method in ("pca", "tsne"):
    inspect_latent_space(
        states, state_labels,
        config=cfg3.with_method(method),
        show_ellipsoids=True,
        show_convex_hulls=(method == "pca"),
        class_names=state_names,
        title=f"Market Regime Embeddings — {method.upper()}",
    ).show()